# Acomodo mapper

## Librerías

In [2]:
import pandas as pd
import numpy as np
import random

import plotly
import requests
from bs4 import BeautifulSoup
import ruptures as rpt
import yfinance as yf

import plotly.graph_objects as go
import matplotlib.pyplot as plt

import kmapper as km
import subprocess
import sys
import networkx as nx
subprocess.check_call([sys.executable, "-m", "pip", "install", "plotly", "nbformat>=4.2.0"])
import sklearn.cluster

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import pairwise_distances
from sklearn.cluster import DBSCAN, KMeans
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots


## Recolección de datos

### Sacar tickers S&P500

In [3]:

# URL de la lista del S&P 500
url = "https://www.slickcharts.com/sp500"


# Hacer la petición con headers para evitar bloqueos
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
}
response = requests.get(url, headers=headers)
response.raise_for_status()  # Raise exception for bad status codes

# Parsear el HTML
soup = BeautifulSoup(response.text, 'html.parser')

# Encontrar la tabla
table = soup.find('table')

if table:
    # Leer la tabla con pandas
    df = pd.read_html(str(table))[0]
    
    # Extraer los tickers
    if 'Symbol' in df.columns:
        tickers = df['Symbol'].tolist()
        print(f"Found {len(tickers)} tickers")
        print("First 10 tickers:", tickers[:10])
    else:
        print("Available columns:", df.columns.tolist())

    # Extraer los pesos
    if 'Weight' in df.columns:
        weights = df['Weight'].tolist()
        print(f"Found {len(weights)} weights")
        print("First 10 weights:", weights[:10])
    else:
        print("Available columns:", df.columns.tolist())
else:
    print("No table found on the page")

# En la lista de ticker, reemplazar los "." por "-"
tickers = [ticker.replace('.', '-') for ticker in tickers]
print("Tickers after replacement:", tickers[:10])

Found 503 tickers
First 10 tickers: ['NVDA', 'AAPL', 'MSFT', 'AMZN', 'AVGO', 'GOOGL', 'META', 'GOOG', 'TSLA', 'BRK.B']
Found 503 weights
First 10 weights: ['8.02%', '6.66%', '6.30%', '4.33%', '2.87%', '2.86%', '2.73%', '2.67%', '2.38%', '1.66%']
Tickers after replacement: ['NVDA', 'AAPL', 'MSFT', 'AMZN', 'AVGO', 'GOOGL', 'META', 'GOOG', 'TSLA', 'BRK-B']


### Información financiera

In [4]:
# =======================================================
# 📊 RECOPILACIÓN DE INFORMACIÓN FUNDAMENTAL DE TICKERS
# =======================================================

print("🔍 Recopilando información detallada de todos los tickers del S&P 500...")
print("=" * 80)

ticker_info_db = {}
failed_tickers = []


for idx, ticker in enumerate(tickers, 1):
    try:
        stock = yf.Ticker(ticker)
        info = stock.info
        
        # Extraer información relevante
        ticker_info_db[ticker] = {
            # Información básica
            'sector': info.get('sector', 'Unknown'),
            'industry': info.get('industry', 'Unknown'),
            'market_cap': info.get('marketCap', None),
            'country': info.get('country', 'Unknown'),
            'full_name': info.get('longName', ticker),
            
            # Métricas de valoración
            'pe_ratio': info.get('trailingPE', None),
            'forward_pe': info.get('forwardPE', None),
            'peg_ratio': info.get('pegRatio', None),
            'price_to_book': info.get('priceToBook', None),
            'price_to_sales': info.get('priceToSalesTrailing12Months', None),
            'enterprise_value': info.get('enterpriseValue', None),
            'ev_to_ebitda': info.get('enterpriseToEbitda', None),
            
            # Dividendos
            'dividend_yield': info.get('dividendYield', None),
            'dividend_rate': info.get('dividendRate', None),
            'payout_ratio': info.get('payoutRatio', None),
            'five_year_avg_dividend_yield': info.get('fiveYearAvgDividendYield', None),
            
            # Rentabilidad
            'profit_margins': info.get('profitMargins', None),
            'operating_margins': info.get('operatingMargins', None),
            'gross_margins': info.get('grossMargins', None),
            'roe': info.get('returnOnEquity', None),
            'roa': info.get('returnOnAssets', None),
            
            # Crecimiento
            'revenue_growth': info.get('revenueGrowth', None),
            'earnings_growth': info.get('earningsGrowth', None),
            'earnings_quarterly_growth': info.get('earningsQuarterlyGrowth', None),
            
            # Riesgo y volatilidad
            'beta': info.get('beta', None),
            '52week_high': info.get('fiftyTwoWeekHigh', None),
            '52week_low': info.get('fiftyTwoWeekLow', None),
            '52week_change': info.get('52WeekChange', None),
            
            # Liquidez y deuda
            'current_ratio': info.get('currentRatio', None),
            'quick_ratio': info.get('quickRatio', None),
            'debt_to_equity': info.get('debtToEquity', None),
            'total_debt': info.get('totalDebt', None),
            'total_cash': info.get('totalCash', None),
            
            # Analistas y recomendaciones
            'recommendation': info.get('recommendationKey', 'Unknown'),
            'recommendation_mean': info.get('recommendationMean', None),
            'target_mean_price': info.get('targetMeanPrice', None),
            'target_high_price': info.get('targetHighPrice', None),
            'target_low_price': info.get('targetLowPrice', None),
            'number_of_analysts': info.get('numberOfAnalystOpinions', None),
            
            # Información adicional
            'employees': info.get('fullTimeEmployees', None),
            'exchange': info.get('exchange', 'Unknown'),
            'quote_type': info.get('quoteType', 'Unknown'),
        }
        
        
    except Exception as e:
        failed_tickers.append(ticker)
        ticker_info_db[ticker] = {
            'sector': 'Unknown',
            'industry': 'Unknown',
            'error': str(e)
        }

print(f"\n{'='*80}")
print(f"📊 RESUMEN DE RECOPILACIÓN:")
if failed_tickers:
    print(f"   Tickers fallidos: {', '.join(failed_tickers[:10])}{'...' if len(failed_tickers) > 10 else ''}")

# Crear DataFrame para análisis fácil
ticker_info_df = pd.DataFrame.from_dict(ticker_info_db, orient='index')

print(f"\n DataFrame de información creado: {ticker_info_df.shape}")


# Guardar en CSV para uso posterior
ticker_info_df.to_csv('sp500_ticker_info_database.csv')
print("\n Base de datos guardada en 'sp500_ticker_info_database.csv'")

# Mostrar primeras filas
print("\n Primeras filas de la base de datos:")
display(ticker_info_df.head(10))

🔍 Recopilando información detallada de todos los tickers del S&P 500...

📊 RESUMEN DE RECOPILACIÓN:

 DataFrame de información creado: (503, 42)

 Base de datos guardada en 'sp500_ticker_info_database.csv'

 Primeras filas de la base de datos:


,sector,industry,market_cap,country,full_name,pe_ratio,forward_pe,peg_ratio,price_to_book,price_to_sales,...,total_cash,recommendation,recommendation_mean,target_mean_price,target_high_price,target_low_price,number_of_analysts,employees,exchange,quote_type
NVDA,Technology,Semiconductors,4939762892800,United States,NVIDIA Corporation,57.639206,49.245148,None,49.328957,29.898457,...,5.679100e+10,strong_buy,1.34375,225.49614,320.0,100.0,57.0,36000.0,NMS,EQUITY
AAPL,Technology,Consumer Electronics,4027681603584,United States,Apple Inc.,41.246200,32.659443,None,61.250280,9.856669,...,5.537200e+10,buy,2.04167,256.00660,320.0,180.0,41.0,150000.0,NMS,EQUITY
MSFT,Technology,Software - Infrastructure,3908020207616,United States,Microsoft Corporation,37.340910,35.167892,None,11.379102,13.871804,...,9.456500e+10,strong_buy,1.21053,622.81885,730.0,483.0,52.0,228000.0,NMS,EQUITY
AMZN,Consumer Cyclical,Internet Retail,2376782315520,United States,"Amazon.com, Inc.",33.972560,36.237396,None,7.117626,3.547235,...,9.318000e+10,strong_buy,1.28986,267.92062,306.0,230.0,63.0,1546000.0,NMS,EQUITY
AVGO,Technology,Semiconductors,1777828757504,United States,Broadcom Inc.,96.530770,61.016210,None,6.357036,29.667068,...,1.110500e+10,strong_buy,1.28261,392.38165,475.0,273.4,41.0,37000.0,NMS,EQUITY
GOOGL,Communication Services,Internet Content & Information,3407559262208,United States,Alphabet Inc.,27.786774,31.415180,None,10.573211,9.174929,...,9.514800e+10,buy,1.50746,297.28537,350.0,185.0,52.0,187103.0,NMS,EQUITY
META,Communication Services,Internet Content & Information,1674266935296,United States,"Meta Platforms, Inc.",29.450730,26.342688,None,9.246896,9.363700,...,4.707100e+10,strong_buy,1.41176,850.59880,1117.0,616.0,60.0,75945.0,NMS,EQUITY
GOOG,Communication Services,Internet Content & Information,3405202325504,United States,Alphabet Inc.,27.828232,31.497208,None,10.588986,9.168582,...,9.514800e+10,strong_buy,1.50000,292.10000,345.0,185.0,17.0,187103.0,NMS,EQUITY
TSLA,Consumer Cyclical,Auto Manufacturers,1463693082624,United States,"Tesla, Inc.",307.762270,135.833330,None,18.293291,15.305314,...,4.164700e+10,hold,2.61702,391.31805,600.0,120.0,41.0,125665.0,NMS,EQUITY
BRK-B,Financial Services,Insurance - Diversified,1031941586944,United States,Berkshire Hathaway Inc.,16.410150,23.830677,None,0.001031,2.787894,...,3.440910e+11,none,NaN,519.66670,593.0,479.0,3.0,392400.0,NYQ,EQUITY


### Datos históricos separados

In [5]:

START_DATE='2025-04-01'
END_DATE='2025-10-08'
AN_DATE='2025-08-01'

In [6]:
# -----------------------------
# Descargar precios históricos
# -----------------------------
# -----------------------------
# Descargar precios históricos
# -----------------------------
all_tickers_data = yf.download(tickers, start=START_DATE, end=END_DATE, auto_adjust=True)['Close']
all_tickers_data = all_tickers_data.dropna(axis=1)  # eliminar columnas con datos faltantes

# DIVIDIR EN DOS PERIODOS
# Período de análisis: desde START_DATE hasta AN_DATE
analysis_data = all_tickers_data.loc[START_DATE:AN_DATE]
tickers_data = analysis_data  # Datos para entrenamiento/análisis

# Período de validación: desde AN_DATE hasta END_DATE
validation_data = all_tickers_data.loc[AN_DATE:END_DATE]  # Datos para validación

print(f"✅ Datos descargados:")
print(f"   📊 Total: {all_tickers_data.shape} (desde {START_DATE} hasta {END_DATE})")
print(f"   📊 Período de análisis: {tickers_data.shape} (desde {START_DATE} hasta {AN_DATE})")
print(f"   📊 Período de validación: {validation_data.shape} (desde {AN_DATE} hasta {END_DATE})")

[*********************100%***********************]  503 of 503 completed

1 Failed download:
['EA']: Timeout('Failed to perform, curl: (28) Operation timed out after 10006 milliseconds with 0 bytes received. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.')


✅ Datos descargados:
   📊 Total: (131, 502) (desde 2025-04-01 hasta 2025-10-08)
   📊 Período de análisis: (85, 502) (desde 2025-04-01 hasta 2025-08-01)
   📊 Período de validación: (47, 502) (desde 2025-08-01 hasta 2025-10-08)


In [7]:
## Datos S&P500
# Descargar SPY para benchmark
SPY_data = yf.download('SPY', start=START_DATE, end=END_DATE, auto_adjust=True)['Close']

# Verificar si es Serie o DataFrame y convertir apropiadamente
if isinstance(SPY_data, pd.Series):
    SPY = SPY_data.to_frame(name='SPY')
else:
    # Si ya es DataFrame, asegurarse que la columna se llame 'SPY'
    SPY = SPY_data.to_frame() if len(SPY_data.shape) == 1 else SPY_data
    if 'SPY' not in SPY.columns:
        SPY.columns = ['SPY']

# Dividir en períodos
SPY_analysis = SPY.loc[START_DATE:AN_DATE]
SPY_validation = SPY.loc[AN_DATE:END_DATE]

print(f"✅ SPY descargado:")
print(f"   📊 SPY_analysis: {SPY_analysis.shape}")
print(f"   📊 SPY_validation: {SPY_validation.shape}")
print(f"   📋 Columnas: {SPY_analysis.columns.tolist()}")

[*********************100%***********************]  1 of 1 completed

✅ SPY descargado:
   📊 SPY_analysis: (85, 1)
   📊 SPY_validation: (47, 1)
   📋 Columnas: ['SPY']


### Datos de métricas

In [8]:
# =======================================================
# CREAR SUBCARPETA PARA VISUALIZACIONES
# =======================================================

import os

# Crear la subcarpeta para todas las visualizaciones
output_folder = 'Mapper_final'
os.makedirs(output_folder, exist_ok=True)
print(f"📁 Subcarpeta creada: {output_folder}/")
print(f"🌐 Todas las visualizaciones HTML se guardarán en esta carpeta")

📁 Subcarpeta creada: Mapper_final/
🌐 Todas las visualizaciones HTML se guardarán en esta carpeta


In [9]:
# Sacamos métricas de cada ticker

financial_metrics = {}
print("\n🔄 Calculando métricas financieras...")
for ticker in tickers_data.columns:
    try:
        prices = tickers_data[ticker].dropna()
        returns = prices.pct_change().dropna()
        
        # Métricas básicas
        total_return = (prices.iloc[-1] / prices.iloc[0]) - 1
        volatility = returns.std() * np.sqrt(252)  # Anualizada
        sharpe = (returns.mean() * 252) / (returns.std() * np.sqrt(252)) if returns.std() > 0 else 0
        
        # Métricas de riesgo
        max_drawdown = ((prices / prices.cummax()) - 1).min()
        var_95 = returns.quantile(0.05)  # Value at Risk 95%
        skewness = returns.skew()
        kurtosis = returns.kurtosis()
        
        # Métricas de tendencia
        returns_positive_ratio = (returns > 0).mean()
        trend_slope = np.polyfit(range(len(prices)), prices.values, 1)[0]
        
        financial_metrics[ticker] = {
            'total_return': total_return,
            'volatility': volatility,
            'sharpe_ratio': sharpe,
            'max_drawdown': max_drawdown,
            'var_95': var_95,
            'skewness': skewness,
            'kurtosis': kurtosis,
            'positive_ratio': returns_positive_ratio,
            'trend_slope': trend_slope / prices.iloc[0]  # Normalizado
        }
    except Exception as e:
        print(f"❌ Error procesando {ticker}: {e}")

# Convertir a DataFrame
metrics_df = pd.DataFrame(financial_metrics).T

# Normalizar datos
scaler = StandardScaler()
metrics_scaled = scaler.fit_transform(metrics_df.fillna(0))# Datos normalizados que ya tenemos

# Preparar datos para KeplerMapper (usar métricas financieras)

ticker_names = metrics_df.index.tolist()


print(f"📊 Datos preparados: {metrics_scaled.shape[0]} tickers, {metrics_scaled.shape[1]} métricas")


🔄 Calculando métricas financieras...
📊 Datos preparados: 502 tickers, 9 métricas


In [10]:
metrics_df

,total_return,volatility,sharpe_ratio,max_drawdown,var_95,skewness,kurtosis,positive_ratio,trend_slope
A,-0.002727,0.391876,0.172752,-0.143978,-0.041324,0.050451,2.178605,0.523810,0.001788
AAPL,-0.092050,0.438495,-0.448208,-0.229890,-0.038698,1.487230,12.495893,0.535714,0.000568
ABBV,-0.036574,0.312671,-0.200549,-0.167438,-0.031001,-1.116517,2.402987,0.547619,0.000490
ABNB,0.056445,0.425379,0.592635,-0.139332,-0.034630,1.515823,10.745959,0.523810,0.002538
ABT,-0.024576,0.254078,-0.165413,-0.123327,-0.019748,-1.978507,10.320301,0.559524,0.000124
...,...,...,...,...,...,...,...,...,...
XYZ,0.336040,0.634249,1.697565,-0.218771,-0.051021,-1.201008,9.420311,0.642857,0.005643
YUM,-0.080839,0.241945,-0.922773,-0.138813,-0.019506,-1.765971,9.879517,0.488095,-0.000043
ZBH,-0.191466,0.314612,-1.863122,-0.199272,-0.023120,-2.313721,12.537985,0.535714,-0.001169
ZBRA,0.184301,0.555862,1.192548,-0.257433,-0.030912,-0.421844,10.661268,0.535714,0.004673


In [11]:
metrics_scaled

array([[-0.31537539,  0.09666923, -0.22014885, ..., -0.93764187,
        -0.04988756, -0.06938036],
       [-0.72432649,  0.44366474, -0.68117995, ...,  1.28207941,
         0.16979274, -0.48179868],
       [-0.47034077, -0.49286026, -0.4973062 , ..., -0.88936707,
         0.38947303, -0.50817704],
       ...,
       [-1.17948505, -0.47841377, -1.73168203, ...,  1.29113533,
         0.16979274, -1.06909307],
       [ 0.54089725,  1.31723769,  0.53699752, ...,  0.88736746,
         0.16979274,  0.90591667],
       [-0.68571131, -0.5582914 , -0.87639734, ..., -1.07529872,
         0.38947303, -0.65764549]], shape=(502, 9))

## Creación de mappers

### Mappers con datos completos

In [12]:
# Escalamos
data = StandardScaler().fit_transform(tickers_data.T)

# Inicializamos Mapper
mapper = km.KeplerMapper(verbose=1)  # verbose=1 para ver debug info
# Proyección (lens): PCA a 2D
lens = mapper.fit_transform(data, projection=PCA(n_components=2))

# Construcción del grafo con KMeans
graph_hist_1 = mapper.map(
    lens, 
    data, 
    clusterer=KMeans(n_clusters=3, random_state=0),  # ✅ KMeans más confiable
    cover=km.Cover(n_cubes=10, perc_overlap=0.5)
)

print(f"✅ Nodos encontrados: {len(graph_hist_1['nodes'])}")
print(f"✅ Aristas encontradas: {len(graph_hist_1['links'])}")

# Visualización interactiva
mapper.visualize(graph_hist_1, 
                 title="Mapper de series de tiempo",
                 custom_tooltips=np.array(ticker_names),  # Mostrar nombres de tickers
                 path_html="Mapper_final/mapper_output.html")

KeplerMapper(verbose=1)
..Composing projection pipeline of length 1:
	Projections: PCA(n_components=2)
	Distance matrices: False
	Scalers: MinMaxScaler()
..Projecting on data shaped (502, 85)

..Projecting data using: 
	PCA(n_components=2)


..Scaling with: MinMaxScaler()

Mapping on data shaped (502, 85) using lens shaped (502, 2)

Creating 100 hypercubes.

Created 76 edges and 33 nodes in 0:00:01.461748.
✅ Nodos encontrados: 33
✅ Aristas encontradas: 26
Wrote visualization to: Mapper_final/mapper_output.html


'<!DOCTYPE html>\n<html>\n\n<head>\n  <meta charset="utf-8">\n  <meta name="generator" content="KeplerMapper">\n  <title>Mapper de series de tiempo | KeplerMapper</title>\n\n  <link rel="icon" type="image/png" href="http://i.imgur.com/axOG6GJ.jpg" />\n\n  <link href=\'https://fonts.googleapis.com/css?family=Roboto+Mono:700,300\' rel=\'stylesheet\' type=\'text/css\'>\n  <style>* {\n  margin: 0;\n  padding: 0;\n}\n\nhtml, body {\n  height: 100%;\n}\n\nbody {\n  font-family: "Roboto Mono", "Helvetica", sans-serif;\n  font-size: 14px;\n}\n\n#logo {\n  width:  85px;\n  height: 85px;\n}\n\n#display {\n  color: #95A5A6;\n  background: #212121;\n}\n\n#header {\n  background: #111111;\n}\n\n#print {\n  color: #000;\n  background: #FFF;\n}\n\nh1 {\n  font-size: 21px;\n  font-weight: 300;\n  font-weight: 300;\n}\n\nh2 {\n  font-size: 18px;\n  padding-bottom: 20px;\n  font-weight: 300;\n}\n\nh3 {\n  font-size: 14px;\n  font-weight: 700;\n  text-transform: uppercase;\n}\n\nh4 {\n  font-size: 13px;\

### Mappers con métricas

In [13]:
# 1. Crear el mapper
mapper = km.KeplerMapper(verbose=1)

KeplerMapper(verbose=1)


In [14]:
# Configuración 1: Proyección por Sharpe
projected_data_sharpe = mapper.fit_transform(metrics_scaled, projection=[2])  # Sharpe ratio (índice 2)
covering = km.Cover(n_cubes=6, perc_overlap=0.5)
G = mapper.map(projected_data_sharpe, metrics_scaled, 
               clusterer=sklearn.cluster.KMeans(n_clusters=2),
               cover=covering)

mapper.visualize(G, 
                title='Mapper: Agrupación por Sharpe',
                color_values=metrics_df['sharpe_ratio'].values,
                color_function_name='Sharpe Ratio',
                node_color_function=np.array(['average','std','sum','max','min']),
                custom_tooltips=np.array(ticker_names),  # Mostrar nombres de tickers
                path_html='Mapper_final/mapper_principal_sharpe.html')  # Guardado en subcarpeta

print(f"Nodos: {len(G['nodes'])}, Aristas: {len(G['links'])}")

..Composing projection pipeline of length 1:
	Projections: [2]
	Distance matrices: False
	Scalers: MinMaxScaler()
..Projecting on data shaped (502, 9)

..Projecting data using: [2]

..Scaling with: MinMaxScaler()

Mapping on data shaped (502, 9) using lens shaped (502, 1)

Creating 6 hypercubes.

Created 15 edges and 12 nodes in 0:00:00.024320.
Wrote visualization to: Mapper_final/mapper_principal_sharpe.html
Nodos: 12, Aristas: 10


In [15]:
# Configuración 2: Proyección por Volatilidad

projected_data_vol = mapper.fit_transform(metrics_scaled, projection=[1])  # Volatilidad (índice 1)
covering_vol = km.Cover(n_cubes=8, perc_overlap=0.4)
G_vol = mapper.map(projected_data_vol, metrics_scaled, 
                   clusterer=sklearn.cluster.KMeans(n_clusters=3),
                   cover=covering_vol)

mapper.visualize(G_vol, 
                title='Mapper: Agrupación por Volatilidad',
                color_values=metrics_df['volatility'].values,
                color_function_name='Volatilidad',
                node_color_function=np.array(['average','max']),
                custom_tooltips=np.array(ticker_names),  # Mostrar nombres de tickers
                path_html='visualizaciones_mapper_indice/mapper_config_volatilidad.html')  # Guardado en subcarpeta

print(f"Nodos: {len(G_vol['nodes'])}, Aristas: {len(G_vol['links'])}")

..Composing projection pipeline of length 1:
	Projections: [1]
	Distance matrices: False
	Scalers: MinMaxScaler()
..Projecting on data shaped (502, 9)

..Projecting data using: [1]

..Scaling with: MinMaxScaler()

Mapping on data shaped (502, 9) using lens shaped (502, 1)

Creating 8 hypercubes.

Created 34 edges and 24 nodes in 0:00:00.030882.
Wrote visualization to: visualizaciones_mapper_indice/mapper_config_volatilidad.html
Nodos: 24, Aristas: 21


In [16]:
# Configuración 3: Proyección por Retorno Total

projected_data_ret = mapper.fit_transform(metrics_scaled, projection=[0])  # Retorno total (índice 0)
covering_ret = km.Cover(n_cubes=7, perc_overlap=0.3)
G_ret = mapper.map(projected_data_ret, metrics_scaled, 
                   clusterer=sklearn.cluster.DBSCAN(eps=0.5, min_samples=1),
                   cover=covering_ret)

mapper.visualize(G_ret, 
                title='Mapper: Agrupación por Retorno Total',
                node_color_function=np.array(['average','std']),
                custom_tooltips=np.array(ticker_names),  # Mostrar nombres de tickers
                path_html='visualizaciones_mapper_indice/mapper_config_retorno_total.html')  # Guardado en subcarpeta

print(f"Nodos: {len(G_ret['nodes'])}, Aristas: {len(G_ret['links'])}")

..Composing projection pipeline of length 1:
	Projections: [0]
	Distance matrices: False
	Scalers: MinMaxScaler()
..Projecting on data shaped (502, 9)

..Projecting data using: [0]

..Scaling with: MinMaxScaler()

Mapping on data shaped (502, 9) using lens shaped (502, 1)

Creating 7 hypercubes.

Created 209 edges and 688 nodes in 0:00:00.083433.
Wrote visualization to: visualizaciones_mapper_indice/mapper_config_retorno_total.html
Nodos: 688, Aristas: 209


In [17]:
# Configuración 4: Proyección Multi-dimensional (PCA)
projected_data_pca = mapper.fit_transform(metrics_scaled, projection='sum')  # Proyección suma
covering_pca = km.Cover(n_cubes=5, perc_overlap=0.6)
G_pca = mapper.map(projected_data_pca, metrics_scaled, 
                   clusterer=sklearn.cluster.KMeans(n_clusters=2),
                   cover=covering_pca)

mapper.visualize(G_pca, 
                title='Mapper: Proyección PCA (Suma)',
                node_color_function=np.array(['min','average']),
                custom_tooltips=np.array(ticker_names),  # Mostrar nombres de tickers
                path_html='visualizaciones_mapper_indice/mapper_config_pca_suma.html')  # Guardado en subcarpeta

..Composing projection pipeline of length 1:
	Projections: sum
	Distance matrices: False
	Scalers: MinMaxScaler()
..Projecting on data shaped (502, 9)

..Projecting data using: sum

..Scaling with: MinMaxScaler()

Mapping on data shaped (502, 9) using lens shaped (502, 1)

Creating 5 hypercubes.

Created 21 edges and 10 nodes in 0:00:00.023242.
Wrote visualization to: visualizaciones_mapper_indice/mapper_config_pca_suma.html


'<!DOCTYPE html>\n<html>\n\n<head>\n  <meta charset="utf-8">\n  <meta name="generator" content="KeplerMapper">\n  <title>Mapper: Proyección PCA (Suma) | KeplerMapper</title>\n\n  <link rel="icon" type="image/png" href="http://i.imgur.com/axOG6GJ.jpg" />\n\n  <link href=\'https://fonts.googleapis.com/css?family=Roboto+Mono:700,300\' rel=\'stylesheet\' type=\'text/css\'>\n  <style>* {\n  margin: 0;\n  padding: 0;\n}\n\nhtml, body {\n  height: 100%;\n}\n\nbody {\n  font-family: "Roboto Mono", "Helvetica", sans-serif;\n  font-size: 14px;\n}\n\n#logo {\n  width:  85px;\n  height: 85px;\n}\n\n#display {\n  color: #95A5A6;\n  background: #212121;\n}\n\n#header {\n  background: #111111;\n}\n\n#print {\n  color: #000;\n  background: #FFF;\n}\n\nh1 {\n  font-size: 21px;\n  font-weight: 300;\n  font-weight: 300;\n}\n\nh2 {\n  font-size: 18px;\n  padding-bottom: 20px;\n  font-weight: 300;\n}\n\nh3 {\n  font-size: 14px;\n  font-weight: 700;\n  text-transform: uppercase;\n}\n\nh4 {\n  font-size: 13p

## Acomodo de indices

In [19]:
graphs={
    'Historic1': graph_hist_1,
    'Sharpe': G,
    'Volatilidad': G_vol,
    'Retorno': G_ret,
    'PCA': G_pca}


graph_names = list(graphs.keys())

categories = {ticker: ticker_info_db[ticker]['sector'] for ticker in ticker_names}
market_caps = {ticker: ticker_info_db[ticker]['market_cap'] for ticker in ticker_names}

In [22]:
"""
Analiza todos los clusters encontrados en los diferentes archivos Mapper
"""
print("📊 ANALIZANDO TODOS LOS CLUSTERS DEL MAPPER")
print("=" * 60)

# Diccionario para almacenar todos los clusters encontrados
def extract_clusters_from_all_graphs(graphs_dict, ticker_names, min_cluster_size=2):
    """
    Extrae clusters de todos los grafos en el diccionario graphs.
    
    Parámetros:
    -----------
    graphs_dict : dict
        Diccionario con {graph_label: graph_object}
    ticker_names : list
        Lista de nombres de tickers
    min_cluster_size : int
        Tamaño mínimo de cluster (default: 2)
    
    Retorna:
    --------
    all_clusters : dict
        Diccionario con todos los clusters encontrados
    """
    all_clusters = {}
    total_clusters = 0
    
    for graph_label, graph in graphs_dict.items():
        print(f"\n🔍 Extrayendo clusters del análisis por {graph_label}:")
        print(f"   Total de nodos: {len(graph['nodes'])}")
        
        clusters_in_graph = 0
        for i, (node_name, node_indices) in enumerate(graph['nodes'].items()):
            if len(node_indices) >= min_cluster_size:
                cluster_tickers = [ticker_names[idx] for idx in node_indices]
                cluster_name = f"{graph_label}_Cluster_{i+1}"
                all_clusters[cluster_name] = {
                    'tickers': cluster_tickers,
                    'type': graph_label,
                    'node_name': node_name,
                    'size': len(cluster_tickers)
                }
                print(f"     -{cluster_name}: {cluster_tickers} ({len(cluster_tickers)} tickers)")
                clusters_in_graph += 1
        
        print(f"   📌 {clusters_in_graph} clusters encontrados en {graph_label}")
        total_clusters += clusters_in_graph
    
    print(f"\n{'='*60}")
    print(f"📋 RESUMEN TOTAL: {total_clusters} clusters encontrados en {len(graphs_dict)} grafos")
    print(f"{'='*60}")
    
    return all_clusters


# Ejecutar la extracción para todos los grafos
all_clusters = extract_clusters_from_all_graphs(
    graphs_dict=graphs,
    ticker_names=ticker_names,
    min_cluster_size=4
)

📊 ANALIZANDO TODOS LOS CLUSTERS DEL MAPPER

🔍 Extrayendo clusters del análisis por Historic1:
   Total de nodos: 33
     -Historic1_Cluster_4: ['AXON', 'BLK', 'GS', 'INTU', 'KLAC', 'META', 'MPWR', 'NOW', 'PH', 'URI'] (10 tickers)
     -Historic1_Cluster_5: ['ADI', 'AMD', 'AVGO', 'BA', 'CEG', 'COIN', 'DASH', 'DDOG', 'DELL', 'GE', 'HOOD', 'HWM', 'JBL', 'JPM', 'MU', 'NRG', 'NVDA', 'ORCL', 'PLTR', 'RCL', 'RL', 'STX', 'TEL', 'TXN', 'VST'] (25 tickers)
     -Historic1_Cluster_6: ['APP', 'CAT', 'CRWD', 'EME', 'ETN', 'GEV', 'HUBB', 'IDXX', 'MSFT', 'PWR', 'ROK', 'SNPS', 'TSLA', 'TT', 'ULTA', 'ZBRA'] (16 tickers)
     -Historic1_Cluster_7: ['A', 'ABNB', 'ABT', 'ACGL', 'ADM', 'AEE', 'AEP', 'AES', 'AFL', 'AIG', 'AKAM', 'ALB', 'ALLE', 'AMCR', 'AMD', 'ANET', 'AOS', 'APA', 'APH', 'APO', 'APTV', 'ARE', 'ATO', 'AWK', 'BAC', 'BALL', 'BAX', 'BBY', 'BEN', 'BF-B', 'BG', 'BIIB', 'BK', 'BKR', 'BLDR', 'BMY', 'BRO', 'BSX', 'BX', 'BXP', 'C', 'CAG', 'CAH', 'CARR', 'CBRE', 'CCI', 'CCL', 'CF', 'CFG', 'CHD', 'CHRW'

### Pesos

In [23]:
# Convert market_caps dict to pandas Series
market_caps_series = pd.Series(market_caps)

# Mostrar muestra del DataFrame
print(f"\n📋 Muestra de las capitalizaciones:")
print(market_caps_series.head(10))
print(f"\n📊 Estadísticas de las capitalizaciones:")
print(f"   💰 Capitalización promedio: ${market_caps_series.mean():,.0f}")
print(f"   📈 Capitalización máxima: ${market_caps_series.max():,.0f}")
print(f"   📉 Capitalización mínima: ${market_caps_series.min():,.0f}")


📋 Muestra de las capitalizaciones:
A         40642621440
AAPL    4027681603584
ABBV     403128582144
ABNB      77349847040
ABT      216983027712
ACGL      31830286336
ACN      154569621504
ADBE     142005862400
ADI      114576416768
ADM       29159389184
dtype: int64

📊 Estadísticas de las capitalizaciones:
   💰 Capitalización promedio: $129,751,684,256
   📈 Capitalización máxima: $4,939,762,892,800
   📉 Capitalización mínima: $6,148,401,664


In [24]:
def create_market_cap_weighted_index(cluster_tickers, data, market_caps_dict, start_value=1):
    """
    Crea un índice ponderado por capitalización de mercado para un cluster específico.
    
    Parámetros:
    -----------
    cluster_tickers : list
        Lista de tickers del cluster
    data : pd.DataFrame
        DataFrame con precios históricos (tickers como columnas)
    market_caps_dict : dict
        Diccionario con {ticker: market_cap}
    start_value : float
        Valor inicial del índice (default: 1)
    
    Retorna:
    --------
    index_series : pd.Series
        Serie temporal del índice
    weights : dict
        Diccionario con los pesos de cada ticker
    """
    # Filtrar tickers que existen tanto en data como en market_caps
    valid_tickers = [t for t in cluster_tickers if t in data.columns and t in market_caps_dict]
    
    if not valid_tickers:
        print(f"⚠️  No hay tickers válidos en el cluster")
        return None, None
    
    # Calcular capitalización total del cluster
    total_market_cap = sum([market_caps_dict[ticker] for ticker in valid_tickers])
    
    # Calcular pesos de cada ticker (capitalización / capitalización total)
    weights = {ticker: market_caps_dict[ticker] / total_market_cap for ticker in valid_tickers}
    
    # Crear el índice
    index_series = pd.Series(0, index=data.index)
    
    for ticker, weight in weights.items():
        # Normalizar precios del ticker al valor inicial
        ticker_prices = data[ticker].dropna()
        if len(ticker_prices) > 0:
            ticker_normalized = (ticker_prices / ticker_prices.iloc[0]) * start_value
            # Agregar al índice con su peso
            index_series = index_series.add(ticker_normalized * weight, fill_value=0)
    
    return index_series, weights


def create_all_cluster_indices(all_clusters, data, market_caps_dict, start_value=1):
    """
    Crea índices ponderados por capitalización para todos los clusters.
    
    Parámetros:
    -----------
    all_clusters : dict
        Diccionario con todos los clusters {cluster_name: {tickers: [...], ...}}
    data : pd.DataFrame
        DataFrame con precios históricos
    market_caps_dict : dict
        Diccionario con {ticker: market_cap}
    start_value : float
        Valor inicial del índice (default: 1)
    
    Retorna:
    --------
    cluster_indices : dict
        Diccionario con {cluster_name: {index: serie, weights: dict, metrics: dict}}
    """
    cluster_indices = {}
    
    
    
    for cluster_name, cluster_info in all_clusters.items():
        cluster_tickers = cluster_info['tickers']
        cluster_type = cluster_info['type']
        
        
        
        # Crear índice para este cluster
        index_series, weights = create_market_cap_weighted_index(
            cluster_tickers, data, market_caps_dict, start_value
        )
        
        if index_series is not None:
            # Calcular métricas del índice
            returns = index_series.pct_change().dropna()
            
            total_return = (index_series.iloc[-1] / index_series.iloc[0] - 1) * 1
            volatility = returns.std() * np.sqrt(252) * 1
            sharpe = (returns.mean() / returns.std() * np.sqrt(252)) if returns.std() > 0 else 0
            max_drawdown = ((index_series / index_series.cummax()) - 1).min() * 1
            
            metrics = {
                'retorno_total': total_return,
                'volatilidad': volatility,
                'sharpe_ratio': sharpe,
                'max_drawdown': max_drawdown,
                'valor_final': index_series.iloc[-1],
                'num_tickers': len(weights)
            }
            
            cluster_indices[cluster_name] = {
                'index': index_series,
                'weights': weights,
                'metrics': metrics,
                'type': cluster_type
            }
            
            
            
            # Mostrar los 3 tickers con mayor peso
            sorted_weights = sorted(weights.items(), key=lambda x: x[1], reverse=True)
            
        else:
            print(f"   ❌ No se pudo crear índice")
    
    
    
    return cluster_indices


def plot_cluster_indices(cluster_indices, benchmark_ticker='SPY', data=None, top_n=10):
    """
    Visualiza los índices de los clusters comparados con un benchmark.
    
    Parámetros:
    -----------
    cluster_indices : dict
        Diccionario con los índices de los clusters
    benchmark_ticker : str
        Ticker del benchmark (default: 'SPY')
    data : pd.DataFrame
        DataFrame con precios históricos (para obtener benchmark)
    top_n : int
        Número de mejores índices a resaltar (default: 10)
    """
    import matplotlib.pyplot as plt
    
    fig, axes = plt.subplots(2, 2, figsize=(18, 12))
    fig.suptitle('📊 Análisis de Índices Ponderados por Capitalización de Mercado', 
                 fontsize=16, fontweight='bold')
    
    # Subplot 1: Performance de todos los índices
    ax1 = axes[0, 0]
    for cluster_name, cluster_data in cluster_indices.items():
        ax1.plot(cluster_data['index'], alpha=0.3, linewidth=1)
    
    # Agregar benchmark si está disponible
    if data is not None and benchmark_ticker in data.columns:
        benchmark = (data[benchmark_ticker] / data[benchmark_ticker].iloc[0]) * 1
        ax1.plot(benchmark, color='red', linewidth=2, label=f'{benchmark_ticker} (Benchmark)', linestyle='--')
    
    ax1.set_title('Performance de Todos los Índices', fontweight='bold')
    ax1.set_xlabel('Fecha')
    ax1.set_ylabel('Valor del Índice (Base 1)')
    ax1.grid(True, alpha=0.3)
    ax1.legend()
    
    # Subplot 2: Top N índices por Sharpe Ratio
    ax2 = axes[0, 1]
    sorted_by_sharpe = sorted(cluster_indices.items(), 
                              key=lambda x: x[1]['metrics']['sharpe_ratio'], 
                              reverse=True)[:top_n]
    
    colors = plt.cm.viridis(np.linspace(0, 1, len(sorted_by_sharpe)))
    for i, (cluster_name, cluster_data) in enumerate(sorted_by_sharpe):
        ax2.plot(cluster_data['index'], label=f"{cluster_name}", 
                color=colors[i], linewidth=2, alpha=0.8)
    
    if data is not None and benchmark_ticker in data.columns:
        benchmark = (data[benchmark_ticker] / data[benchmark_ticker].iloc[0]) * 1
        ax2.plot(benchmark, color='red', linewidth=2, label=f'{benchmark_ticker}', linestyle='--')
    
    ax2.set_title(f'Top {top_n} Índices por Sharpe Ratio', fontweight='bold')
    ax2.set_xlabel('Fecha')
    ax2.set_ylabel('Valor del Índice (Base 100)')
    ax2.grid(True, alpha=0.3)
    ax2.legend(fontsize=8, loc='best')
    
    # Subplot 3: Retorno vs Volatilidad (Scatter)
    ax3 = axes[1, 0]
    returns = [data['metrics']['retorno_total'] for data in cluster_indices.values()]
    vols = [data['metrics']['volatilidad'] for data in cluster_indices.values()]
    sharpes = [data['metrics']['sharpe_ratio'] for data in cluster_indices.values()]
    
    scatter = ax3.scatter(vols, returns, c=sharpes, s=100, alpha=0.6, 
                         cmap='RdYlGn', edgecolors='black', linewidth=1)
    
    # Añadir benchmark si está disponible
    if data is not None and benchmark_ticker in data.columns:
        bench_returns = (data[benchmark_ticker].pct_change().dropna())
        bench_ret = (data[benchmark_ticker].iloc[-1] / data[benchmark_ticker].iloc[0] - 1) * 1
        bench_vol = bench_returns.std() * np.sqrt(252) * 1
        ax3.scatter(bench_vol, bench_ret, color='red', s=200, marker='*', 
                   edgecolors='black', linewidth=2, label=benchmark_ticker, zorder=5)
    
    ax3.set_title('Retorno vs Volatilidad (Frontera Eficiente)', fontweight='bold')
    ax3.set_xlabel('Volatilidad (%)')
    ax3.set_ylabel('Retorno Total (%)')
    ax3.grid(True, alpha=0.3)
    plt.colorbar(scatter, ax=ax3, label='Sharpe Ratio')
    ax3.legend()
    
    # Subplot 4: Métricas comparativas (barras)
    ax4 = axes[1, 1]
    cluster_names = list(cluster_indices.keys())[:15]  # Top 15 para legibilidad
    sharpe_values = [cluster_indices[name]['metrics']['sharpe_ratio'] for name in cluster_names]
    
    bars = ax4.barh(range(len(cluster_names)), sharpe_values, 
                    color=plt.cm.RdYlGn(np.linspace(0.3, 0.9, len(cluster_names))))
    ax4.set_yticks(range(len(cluster_names)))
    ax4.set_yticklabels(cluster_names, fontsize=8)
    ax4.set_xlabel('Sharpe Ratio')
    ax4.set_title(f'Comparación de Sharpe Ratios (Top 15)', fontweight='bold')
    ax4.grid(True, alpha=0.3, axis='x')
    
    # Añadir valores en las barras
    for i, bar in enumerate(bars):
        width = bar.get_width()
        ax4.text(width, bar.get_y() + bar.get_height()/2, 
                f'{width:.2f}', ha='left', va='center', fontsize=8)
    
    plt.tight_layout()
    plt.show()
    
    return fig


def export_cluster_indices_to_csv(cluster_indices, filename='cluster_indices.csv'):
    """
    Exporta los índices de todos los clusters a un archivo CSV.
    
    Parámetros:
    -----------
    cluster_indices : dict
        Diccionario con los índices de los clusters
    filename : str
        Nombre del archivo CSV (default: 'cluster_indices.csv')
    """
    # Crear DataFrame con todos los índices
    indices_df = pd.DataFrame()
    
    for cluster_name, cluster_data in cluster_indices.items():
        indices_df[cluster_name] = cluster_data['index']
    
    # Guardar a CSV
    indices_df.to_csv(filename)
    print(f"✅ Índices exportados a: {filename}")
    print(f"   Dimensiones: {indices_df.shape}")
    
    # También crear un archivo con las métricas
    metrics_filename = filename.replace('.csv', '_metrics.csv')
    metrics_data = []
    
    for cluster_name, cluster_data in cluster_indices.items():
        metrics = cluster_data['metrics'].copy()
        metrics['cluster_name'] = cluster_name
        metrics['type'] = cluster_data['type']
        metrics_data.append(metrics)
    
    metrics_df = pd.DataFrame(metrics_data)
    metrics_df = metrics_df[['cluster_name', 'type', 'retorno_total', 'volatilidad', 
                             'sharpe_ratio', 'max_drawdown', 'valor_final', 'num_tickers']]
    metrics_df.to_csv(metrics_filename, index=False)
    print(f"✅ Métricas exportadas a: {metrics_filename}")
    
    return indices_df, metrics_df


print("✅ Funciones de índices ponderados cargadas:")
print("   - create_market_cap_weighted_index()")
print("   - create_all_cluster_indices()")
print("   - plot_cluster_indices()")
print("   - export_cluster_indices_to_csv()")

✅ Funciones de índices ponderados cargadas:
   - create_market_cap_weighted_index()
   - create_all_cluster_indices()
   - plot_cluster_indices()
   - export_cluster_indices_to_csv()


In [25]:
# Paso 1: Verificar que tenemos todos los datos necesarios
print("🔍 Verificando datos necesarios...")
print(f"   ✓ all_clusters: {len(all_clusters)} clusters")
print(f"   ✓ tickers_data: {tickers_data.shape}")
print(f"   ✓ market_caps: {len(market_caps)} tickers con capitalización")

# Paso 2: Crear índices para todos los clusters
cluster_indices = create_all_cluster_indices(
    all_clusters=all_clusters,
    data=tickers_data,
    market_caps_dict=market_caps,
    start_value=1
)

# Paso 3: Crear tabla resumen con métricas
print("\n" + "=" * 100)
print("  RESUMEN DE ÍNDICES CREADOS")
print("=" * 100)

summary_data = []
for cluster_name, cluster_data in cluster_indices.items():
    summary_data.append({
        'Cluster': cluster_name,
        'Tipo': cluster_data['type'],
        'Tickers': cluster_data['metrics']['num_tickers'],
        'Retorno (%)': cluster_data['metrics']['retorno_total'],
        'Volatilidad (%)': cluster_data['metrics']['volatilidad'],
        'Sharpe Ratio': cluster_data['metrics']['sharpe_ratio'],
        'Max Drawdown (%)': cluster_data['metrics']['max_drawdown'],
        'Valor Final': cluster_data['metrics']['valor_final']
    })

summary_df = pd.DataFrame(summary_data)
summary_df = summary_df.sort_values('Sharpe Ratio', ascending=False)

print("\n📊 TOP 10 ÍNDICES POR SHARPE RATIO:")
print(summary_df.head(10).to_string(index=False))

print("\n\n📊 ESTADÍSTICAS GENERALES:")
print(f"   📈 Retorno promedio: {summary_df['Retorno (%)'].mean():.2f}%")
print(f"   📊 Volatilidad promedio: {summary_df['Volatilidad (%)'].mean():.2f}%")
print(f"   ⭐ Sharpe Ratio promedio: {summary_df['Sharpe Ratio'].mean():.2f}")
print(f"   📉 Max Drawdown promedio: {summary_df['Max Drawdown (%)'].mean():.2f}%")

🔍 Verificando datos necesarios...
   ✓ all_clusters: 64 clusters
   ✓ tickers_data: (85, 502)
   ✓ market_caps: 502 tickers con capitalización

  RESUMEN DE ÍNDICES CREADOS

📊 TOP 10 ÍNDICES POR SHARPE RATIO:
               Cluster        Tipo  Tickers  Retorno (%)  Volatilidad (%)  Sharpe Ratio  Max Drawdown (%)  Valor Final
  Historic1_Cluster_19   Historic1        9     0.393299         0.292140      3.552932         -0.078843     1.393299
Volatilidad_Cluster_18 Volatilidad       11     0.728738         0.502964      3.516987         -0.163629     1.728738
     Sharpe_Cluster_12      Sharpe       33     0.367401         0.281814      3.473541         -0.098782     1.367401
        PCA_Cluster_10         PCA       13     0.362216         0.285470      3.391444         -0.076326     1.362216
     Sharpe_Cluster_11      Sharpe       17     0.655900         0.487510      3.346130         -0.150871     1.655900
         PCA_Cluster_9         PCA       56     0.580630         0.449181    